# Retail Flow data generation with Faker

## Table of contents

* **1. Setup and Libraries**: Installing and importing required Python packages.
* **2. Static data definitions**: Defining dictionaries for stores, base zones, cameras, and the product catalog.
* **3. Base entities (Stores, Zones, Cameras)**: Multiplying base configurations across all stores to establish Primary and Foreign Keys.
* **4. Sales & Sold Products simulation**: Generating checkout tickets and mapping products to specific store zones.
* **5. Store traffic & bounce rates**: Simulating hourly visitor distribution while strictly respecting store capacity limits.
* **6. Camera detections simulation**: Generating spatial (x, y) client detections bounded within actual camera zones.
* **7. Data export (CSV)**: Saving the generated dataframes to a local directory for analysis.

### 1. Setup and Libraries
First, we will install the required libraries and import the necessary modules. We also initialize Faker to use Spanish for realistic naming.

In [24]:
# !pip install faker pandas

In [25]:
from faker.providers import DynamicProvider
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd
import random
import uuid
import csv

# limit Faker to Spanish
fake = Faker('es_ES')

### 2. Static data definitions
Here we define the static configuration of our retail environment: the stores, the zones inside them, the camera models, and the product catalog.

In [26]:
stores_dict = {
    'SRCL': {
        'store_name': 'Sara Colon',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        },
    'SRGV': {
        'store_name': 'Sara Gran Via',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        },
    'SRAQ': {
        'store_name': 'Sara Aqua',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        },
    'SRBN': {'store_name': 'Sara Bonaire',
             'city': 'Valencia',
             'm2': 585,
             'max_capacity': 290
             },
    'SRSL': {
        'store_name': 'Sara Saler',
        'city': 'Valencia',
        'm2': 585,
        'max_capacity': 290
        }
}

In [27]:
base_zones = {
    'ENTR': {
        'zone_name': 'Entrance',
        'zone_type': 'Entrance',
        'coord_lims': None
        },
    'WM': {'zone_name': 'Women',
           'zone_type': 'Women clothes',
           'coord_lims': {
               'x_min': 15, 'x_max': 28, 'y_min': 3, 'y_max': 15
               }
           },
    'MN': {
        'zone_name': 'Men',
        'zone_type': 'Men clothes',
        'coord_lims': {
            'x_min': 28, 'x_max': 40, 'y_min': 0, 'y_max': 8
            }
            },
    'KD': {
        'zone_name': 'Kids',
        'zone_type': 'Kid clothes',
        'coord_lims': {
            'x_min': 28, 'x_max': 40, 'y_min': 8, 'y_max': 15
            }
            },
    'CO': {
        'zone_name': 'Checkout',
        'zone_type': 'Checkout',
        'coord_lims': {
            'x_min': 15, 'x_max': 22, 'y_min': 0, 'y_max': 3
            }
            },
    'FR': {
        'zone_name': 'Fitting Rooms',
        'zone_type': 'Fitting Rooms',
        'coord_lims': {
            'x_min': 7, 'x_max': 15, 'y_min': 5, 'y_max': 10
            }
            }
}

In [28]:
base_cameras = {
    'ENTRCM': {
        'zone_id': 'ENTR',
        'model': 'Bullet'
        },
    'WMCM': {
        'zone_id': 'WM',
        'model': 'Fisheye'
        },
    'MNCM': {
        'zone_id': 'MN', 
        'model': 'Fisheye'
        },
    'KDCM': {
        'zone_id': 'KD',
        'model': 'Fisheye'
        },
    'COCM': {
        'zone_id': 'CO',
        'model': 'Turret'
        },
    'FRCM': {
        'zone_id': 'FR',
        'model': 'Dome'
        }
}

In [29]:
# base products w/o a specific ticket_id or zone_id
catalog = [
    {'name': 'Women Shirt', 'category': 'Tops', 'price': 25.99, 'base_zone': 'WM'},
    {'name': 'Women Jeans', 'category': 'Bottoms', 'price': 49.95, 'base_zone': 'WM'},
    {'name': 'Women Summer Dress', 'category': 'Dresses', 'price': 35.50, 'base_zone': 'WM'},
    {'name': 'Women Leather Jacket', 'category': 'Outerwear', 'price': 120.00, 'base_zone': 'WM'},
    {'name': 'Women Wool Sweater', 'category': 'Tops', 'price': 45.00, 'base_zone': 'WM'},

    {'name': 'Men Basic T-Shirt', 'category': 'Tops', 'price': 15.99, 'base_zone': 'MN'},
    {'name': 'Men Chino Trousers', 'category': 'Bottoms', 'price': 39.99, 'base_zone': 'MN'},
    {'name': 'Men Hoodie', 'category': 'Tops', 'price': 45.00, 'base_zone': 'MN'},
    {'name': 'Men Denim Jacket', 'category': 'Outerwear', 'price': 59.95, 'base_zone': 'MN'},
    {'name': 'Men Tailored Suit', 'category': 'Suits', 'price': 250.00, 'base_zone': 'MN'},

    {'name': 'Kids Graphic T-Shirt', 'category': 'Tops', 'price': 12.50, 'base_zone': 'KD'},
    {'name': 'Kids Denim Shorts', 'category': 'Bottoms', 'price': 18.99, 'base_zone': 'KD'},
    {'name': 'Kids Winter Coat', 'category': 'Outerwear', 'price': 45.00, 'base_zone': 'KD'},
    {'name': 'Kids Pajama Set', 'category': 'Nightwear', 'price': 22.50, 'base_zone': 'KD'},
    {'name': 'Kids School Uniform', 'category': 'Sets', 'price': 35.00, 'base_zone': 'KD'}
]

### 3. Base entities (Stores, Zones, Cameras)
We multiply the base zones and cameras across all available stores to generate unique Primary Keys (PK) and maintain Foreign Key (FK) integrity.

In [35]:
# A. STORES
stores_list = [{'store_id': key, **value} for key, value in stores_dict.items()]
df_stores = pd.DataFrame(stores_list)

# B. ZONES & C. CAMERAS
zones_list = []
cameras_list = []

for store_id in stores_dict.keys():
    for zone_id, zone_data in base_zones.items():
        # PK & FK for Zones
        real_zone_id = f"{zone_id}_{store_id}"
        zones_list.append({
            'zone_id': real_zone_id,
            'store_id': store_id,
            'zone_name': zone_data['zone_name'],
            'zone_type': zone_data['zone_type'],
            'coord_lims': str(zone_data['coord_lims']) # string for the CSV
        })
        
    for camera_id, camera_data in base_cameras.items():
        # PK & FK for Cameras
        real_camera_id = f"{camera_id}_{store_id}"
        real_zone_id = f"{camera_data['zone_id']}_{store_id}"
        lims = base_zones[camera_data['zone_id']]['coord_lims']
        
        cameras_list.append({
            'camera_id': real_camera_id,
            'zone_id': real_zone_id,
            'model': camera_data['model'],
            'installation_date': fake.date_between(start_date='-2y', end_date='today').isoformat(),
            'condition': random.choice(['Optimal', 'Optimal', 'Optimal', 'Good', 'Good', 'Needs Maintenance', 'Error']),
            'lims': str(lims)
        })

df_zones = pd.DataFrame(zones_list)
df_cameras = pd.DataFrame(cameras_list)
display(df_zones.head())
display(df_cameras.head())

,zone_id,store_id,zone_name,zone_type,coord_lims
0,ENTR_SRCL,SRCL,Entrance,Entrance,None
1,WM_SRCL,SRCL,Women,Women clothes,"{'x_min': 15, 'x_max': 28, 'y_min': 3, 'y_max'..."
2,MN_SRCL,SRCL,Men,Men clothes,"{'x_min': 28, 'x_max': 40, 'y_min': 0, 'y_max'..."
3,KD_SRCL,SRCL,Kids,Kid clothes,"{'x_min': 28, 'x_max': 40, 'y_min': 8, 'y_max'..."
4,CO_SRCL,SRCL,Checkout,Checkout,"{'x_min': 15, 'x_max': 22, 'y_min': 0, 'y_max'..."


,camera_id,zone_id,model,installation_date,condition,lims
0,ENTRCM_SRCL,ENTR_SRCL,Bullet,2025-08-12,Error,None
1,WMCM_SRCL,WM_SRCL,Fisheye,2024-07-23,Optimal,"{'x_min': 15, 'x_max': 28, 'y_min': 3, 'y_max'..."
2,MNCM_SRCL,MN_SRCL,Fisheye,2024-05-02,Needs Maintenance,"{'x_min': 28, 'x_max': 40, 'y_min': 0, 'y_max'..."
3,KDCM_SRCL,KD_SRCL,Fisheye,2024-12-21,Optimal,"{'x_min': 28, 'x_max': 40, 'y_min': 8, 'y_max'..."
4,COCM_SRCL,CO_SRCL,Turret,2024-06-05,Good,"{'x_min': 15, 'x_max': 22, 'y_min': 0, 'y_max'..."


### 4. Sales & Sold Products simulation
Simulating customer checkouts. Each generated ticket maps purchased products to the specific zone of the store where the sale took place.

In [36]:
# D. SALES & E. SOLDPRODUCTS
sales_list = []
soldproducts_list = []
num_sales = 200 # number of tickets to generate

start_date = datetime(2023, 10, 1)
end_date = datetime(2023, 10, 31)

for _ in range(num_sales):
    ticket_id = f"TKT-{fake.unique.random_int(min=10000, max=99999)}"
    store_id = random.choice(list(stores_dict.keys()))
    sale_time = fake.date_time_between(start_date=start_date, end_date=end_date)
    
    # random selection of products for the ticket
    num_items = random.randint(1, 5)
    cart = random.choices(catalog, k=num_items)
    total_euros = sum(item['price'] for item in cart)
    
    sales_list.append({
        'ticket_id': ticket_id,
        'store_id': store_id,
        'timestamp': sale_time.isoformat(),
        'total_euros': round(total_euros, 2),
        'product_amount': num_items,
        'checkout_number': random.randint(1, 4)
    })
    
    for item in cart:
        soldproducts_list.append({
            'product_id': str(uuid.uuid4())[:8], # unique ID
            'ticket_id': ticket_id,
            'zone_id': f"{item['base_zone']}_{store_id}", # FK that matches the store_id
            'name': item['name'],
            'category': item['category'],
            'price': item['price']
        })

df_sales = pd.DataFrame(sales_list)
df_soldproducts = pd.DataFrame(soldproducts_list)

display(df_sales.head())
display(df_soldproducts.head())

,ticket_id,store_id,timestamp,total_euros,product_amount,checkout_number
0,TKT-32598,SRSL,2023-10-07T04:52:30,120.00,1,2
1,TKT-83303,SRCL,2023-10-02T05:53:45,185.94,3,2
2,TKT-93601,SRSL,2023-10-11T01:10:56,120.00,1,1
3,TKT-18471,SRCL,2023-10-30T10:03:14,245.50,5,1
4,TKT-31517,SRBN,2023-10-30T21:25:40,310.99,3,1


,product_id,ticket_id,zone_id,name,category,price
0,e08ad664,TKT-32598,WM_SRSL,Women Leather Jacket,Outerwear,120.00
1,b2fe56c5,TKT-83303,WM_SRCL,Women Leather Jacket,Outerwear,120.00
2,bd319b6b,TKT-83303,MN_SRCL,Men Basic T-Shirt,Tops,15.99
3,a3dda238,TKT-83303,WM_SRCL,Women Jeans,Bottoms,49.95
4,b0be206b,TKT-93601,WM_SRSL,Women Leather Jacket,Outerwear,120.00


### 5. Store traffic & bounce rates
Simulating store traffic over a 12-hour window. The algorithm ensures the sum of people across all zones never exceeds the store's `max_capacity`.

In [39]:
# F. TRAFFIC

traffic_list = []
traffic_date = datetime(2023, 10, 1, 10, 0) # Starts at 10:00

for store_id, store_info in stores_dict.items():
    max_cap = store_info['max_capacity']
    
    # get zones for this specific store
    store_zones = [z for z in zones_list if z['store_id'] == store_id]
    
    for i in range(12): # 12 hours of operation
        current_time = traffic_date + timedelta(hours=i)
        
        # 1. random number of total visitors that never exceeds the max_capacity
        people_in_store = random.randint(5, max_cap)
        
        # 2. we start with all zones having 0 people
        zone_counts = [0] * len(store_zones)

        for _ in range(people_in_store):
            random_zone_index = random.randint(0, len(store_zones) - 1)
            zone_counts[random_zone_index] += 1

        # 3. save the records
        for zone, count in zip(store_zones, zone_counts):
            
            bounce = round(random.uniform(0.05, 0.60), 2)
            
            # peak is the actual people + a variation
            peak_variance = random.randint(0, int(count * 0.2) + 2)
            logical_peak = count + peak_variance

            traffic_list.append({
                'date_time': current_time.isoformat(),
                'store_id': store_id,
                'zone_id': zone['zone_id'],
                'visitor_count': count, 
                'average_time_in_store': round(random.uniform(2.0, 20.0), 1),
                'peak_people': logical_peak,
                'bounce_rate': bounce
            })

df_traffic = pd.DataFrame(traffic_list)
display(df_traffic.head())

,date_time,store_id,zone_id,visitor_count,average_time_in_store,peak_people,bounce_rate
0,2023-10-01T10:00:00,SRCL,ENTR_SRCL,4,12.5,6,0.50
1,2023-10-01T10:00:00,SRCL,WM_SRCL,3,14.5,5,0.39
2,2023-10-01T10:00:00,SRCL,MN_SRCL,5,17.9,6,0.34
3,2023-10-01T10:00:00,SRCL,KD_SRCL,7,6.0,7,0.43
4,2023-10-01T10:00:00,SRCL,CO_SRCL,4,12.0,6,0.33


### 6. Camera detections simulation
Generating precise spatial detections. Coordinates (x, y) are randomly generated but constrained to the boundaries (`coord_lims`) of the specific camera's zone.

In [40]:
# G. DETECTIONS
detections_list = []
# 2 hour margin
det_start_time = datetime(2023, 10, 1, 12, 0)

for cam in cameras_list:
    current_time = det_start_time
    cam_lims = eval(cam['lims']) if cam['lims'] != 'None' else None
    
    for _ in range(24): # ~2 hours (1 detection every 5 mins = 12/hour)
        # between 5 & 10 minutes
        current_time += timedelta(minutes=random.randint(5, 10))
        
        # generation of a random coodinate between the limits
        coord = None
        if cam_lims:
            x = round(random.uniform(cam_lims['x_min'], cam_lims['x_max']), 2)
            y = round(random.uniform(cam_lims['y_min'], cam_lims['y_max']), 2)
            coord = f"({x}, {y})"
            
        detections_list.append({
            'detection_id': str(uuid.uuid4())[:12],
            'tracking_id': f"TRK-{random.randint(100,999)}",
            'camera_id': cam['camera_id'],
            'timestamp': current_time.isoformat(),
            'class_object': random.choice(['Client', 'Client', 'Client', 'Client', 'Stuff', 'Stuff']),
            'coord_lims': coord,
            'confidence': round(random.uniform(0.70, 0.99), 2)
        })

df_detections = pd.DataFrame(detections_list)
display(df_detections.head())

,detection_id,tracking_id,camera_id,timestamp,class_object,coord_lims,confidence
0,944a617b-11e,TRK-985,ENTRCM_SRCL,2023-10-01T12:08:00,Client,None,0.88
1,b51280be-7da,TRK-914,ENTRCM_SRCL,2023-10-01T12:17:00,Client,None,0.95
2,975f2031-de4,TRK-700,ENTRCM_SRCL,2023-10-01T12:27:00,Client,None,0.88
3,2e57e828-7dc,TRK-218,ENTRCM_SRCL,2023-10-01T12:32:00,Stuff,None,0.92
4,bb3ab435-4d6,TRK-528,ENTRCM_SRCL,2023-10-01T12:42:00,Client,None,0.83


### 7. Data export (CSV)
Finally, we create a `./data` directory (if it doesn't exist) and save all our generated dataframes to CSV files for further analysis.

In [41]:
# 3. export to csv
df_stores.to_csv('./data/stores.csv', index=False)
df_zones.to_csv('./data/zones.csv', index=False)
df_cameras.to_csv('./data/cameras.csv', index=False)
df_sales.to_csv('./data/sales.csv', index=False)
df_soldproducts.to_csv('./data/soldproducts.csv', index=False)
df_traffic.to_csv('./data/traffic.csv', index=False)
df_detections.to_csv('./data/detections.csv', index=False)

print("CSV files succesfully created.")

CSV files succesfully created.
